# Module 2 - Module Demo: Introduction to Generative AI
**Generative AI Fellowship — Beginner**

## The Story
You have been hired by a Lagos startup called **NaijaBiz AI** to build a simple AI-powered business advisor.
You will build it step by step — from your first API call to a fully prompt-engineered assistant.

**What you will build:**
- Connect to an LLM API securely
- Choose the right model for the use case
- Engineer a production-quality system message
- Apply prompt techniques to improve output quality
- Handle LLM limitations responsibly


In [ ]:
# setup
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print("NaijaBiz AI — build session started.")

## Part 1 - First Contact

In [ ]:
# first-contact
# The foundation of every AI product — one API call
response = ask("What are the top three opportunities for tech startups in Nigeria right now?")
print(response)

## Part 2 - Model Selection

In [ ]:
# model-selection
# Decision framework for NaijaBiz AI
print("Model Selection for NaijaBiz AI")
print("=" * 40)
print("Use case: Nigerian business advisor chatbot")
print("Sensitive personal data? No → cloud API is fine")
print("Very long documents? No → standard context window sufficient")
print("Budget-conscious startup? Yes → use cheapest capable model")
print("Code generation needed? No")
print()
print("Decision: GPT-4o-mini")
print("Reason: Fast, affordable, excellent instruction-following")
print()
print("Estimated cost at 500 conversations/day:")
daily_calls = 500
avg_tokens = 600  # ~300 input + 300 output
price_per_1m_input = 0.15
price_per_1m_output = 0.60
daily_cost = (daily_calls * 300 / 1_000_000 * price_per_1m_input) + (daily_calls * 300 / 1_000_000 * price_per_1m_output)
print(f"  Daily: ${daily_cost:.4f}")
print(f"  Monthly: ${daily_cost * 30:.2f}")

## Part 3 - Building the System Message

In [ ]:
# weak-advisor
weak_system = "You are a business advisor."

question = "Should I accept a ₦2 million investment for 40% equity in my early-stage food delivery startup?"

print("Version 1 — Weak system message:")
print(ask(question, system=weak_system))

In [ ]:
# strong-advisor
strong_system = """You are NaijaBiz AI — a business advisor specialising in Nigerian startups and SMEs.

Your expertise:
- Nigerian market conditions, regulations, and business culture
- Startup funding, equity, and valuation in the Nigerian context
- Operations, growth strategies, and challenges specific to Nigeria

Rules:
- Always give practical, Nigeria-specific advice using Naira amounts and local examples
- If you are not certain about a specific regulation or figure, say so clearly
- End every response with one concrete next action the entrepreneur should take
- Keep responses focused and under 200 words unless asked for detail"""

print("Version 2 — Strong system message:")
print(ask(question, system=strong_system))

## Part 4 - Applying Prompt Techniques

In [ ]:
# zero-shot-advice
print("Zero-shot — standard question:")
print(ask(
    "What are the legal requirements to register a tech startup in Nigeria?",
    system=strong_system
))

In [ ]:
# few-shot-advice
few_shot_prompt = """Answer business questions in this exact format:
ANSWER: [direct answer in 1-2 sentences]
WHY IT MATTERS: [one sentence on the Nigerian context]
NEXT STEP: [one concrete action]

Example:
Q: Should I register my business with CAC?
ANSWER: Yes — CAC registration is legally required and unlocks banking, contracts, and investor trust.
WHY IT MATTERS: In Nigeria, unregistered businesses cannot open corporate bank accounts or sign formal contracts.
NEXT STEP: Visit the CAC portal at cac.gov.ng and begin your name reservation today.

Now answer:
Q: What are the legal requirements to register a tech startup in Nigeria?"""

print("Few-shot — structured format:")
print(ask(few_shot_prompt, system=strong_system))

In [ ]:
# cot-advice
maths_question = """My startup has monthly revenue of ₦850,000 and monthly expenses of ₦620,000.
I want to hire a developer at ₦180,000/month. Can I afford it, and how many months of runway do I have
if revenue stays flat?

Think through this step by step."""

print("Chain-of-thought — financial calculation:")
print(ask(maths_question, system=strong_system, temperature=0))

## Part 5 - Handling Limitations

In [ ]:
# knowledge-cutoff-demo
print("Without current data — knowledge cutoff problem:")
print(ask(
    "What is the current CBN interest rate and how should it affect my decision to take a bank loan?",
    system=strong_system
))

In [ ]:
# grounded-advice
current_data = """Current Nigerian economic context (July 2026):
- CBN Monetary Policy Rate: 26.25%
- Average commercial bank lending rate: 28-32%
- Inflation rate: 28.4%
- USD/NGN: ₦1,580"""

grounded_question = f"""{current_data}

Given these conditions, should a Nigerian startup take a bank loan at 30% annual interest to fund expansion?
Think through this step by step."""

print("With current data provided:")
print(ask(grounded_question, system=strong_system, temperature=0))

In [ ]:
# hallucination-guard
safe_system = strong_system + """

IMPORTANT: If you are not certain about a specific regulation, tax rate, or legal requirement,
always say: 'I recommend verifying this with a qualified Nigerian lawyer or accountant.'
Never state specific regulatory figures unless you are certain they are accurate."""

print("With hallucination guard:")
print(ask(
    "What is the exact penalty for late VAT filing in Nigeria and which section of the law covers it?",
    system=safe_system,
    temperature=0
))

## Part 6 - The Complete NaijaBiz Advisor

In [ ]:
# complete-advisor
# Everything combined — the final product
naijabiz_system = """You are NaijaBiz AI — a business advisor specialising in Nigerian startups and SMEs.

Your expertise:
- Nigerian market conditions, regulations, and business culture
- Startup funding, equity, and growth strategy in the Nigerian context
- Practical operations advice for Lagos, Abuja, and other Nigerian cities

Rules:
- Always give practical, Nigeria-specific advice using Naira amounts where relevant
- If uncertain about a specific law, rate, or regulation, say: 'Verify this with a qualified Nigerian professional'
- Never state specific legal or tax figures unless certain they are accurate
- End every response with one concrete next action
- Format: brief answer first, then reasoning, then next step
- Keep responses under 250 words unless asked for detail"""

# Test with multiple business questions
test_questions = [
    "I have a food tech idea but no technical co-founder. Should I learn to code or hire a developer?",
    "My Nigerian e-commerce startup is growing. Should I raise VC funding or stay bootstrapped?",
    "How do I protect my startup idea from being copied in Nigeria?"
]

for q in test_questions:
    print(f"Q: {q}")
    print("-" * 50)
    print(ask(q, system=naijabiz_system))
    print()

## Module 2 Complete

**What you built:** NaijaBiz AI — a prompt-engineered, limitation-aware Nigerian business advisor.

**Skills demonstrated:**
- Secure API access with environment variables
- Model selection based on use case, cost, and privacy
- Comprehensive system message engineering
- Zero-shot, few-shot, and chain-of-thought prompting
- Knowledge cutoff handling with grounded prompts
- Hallucination prevention with uncertainty instructions

**Next: Module 3 — Web Development Fundamentals**
We build the user interface for applications like this one.